# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row equals one page

In [43]:
import pandas as pd
df = pd.read_csv(r'C:\Users\hamto\OneDrive\Desktop\Flyrank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [44]:
df['content_id'].nunique()

30000

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. ### Feature:
    content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, content_type, char_count, competition_level, days_with_sessions

2. ### Label:
    Refresh_score = dampened_trend_pct * (cpc * search_volume)

3. ### Context:
    age_tier, age_tier_order, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier

4. ### excluded:
    impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, cpc, search volume,  sessions_prev_30d, trend_pct, trend_direction, model_used, provider_used. 
        I excluded these features because if they are included, they would cause data leakage. These inputs were one way or the other, used to create or influence the target. Therefore, so that the model does not learn the rigid pattern, we drop them.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [45]:
df['trend_pct'].describe() 

count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

In [46]:
(df['impressions_90d'] >= (df['impressions_prev_30d'] + df['impressions_last_30d'])).value_counts()

True    30000
Name: count, dtype: int64

In [47]:
impressions_diff = df['impressions_90d'] - (df['impressions_prev_30d'] + df['impressions_last_30d'])
impressions_diff.mean()

np.float64(1988.2290666666668)

In [48]:
impressions_diff.describe()



count     30000.000000
mean       1988.229067
std        6099.411747
min           0.000000
25%          28.000000
50%         308.000000
75%        1512.250000
max      258505.000000
dtype: float64

In [49]:
df['impressions_last_30d'].describe()

count     30000.000000
mean       1429.058733
std        5643.852081
min           0.000000
25%          10.000000
50%         139.000000
75%         768.000000
max      238796.000000
Name: impressions_last_30d, dtype: float64

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. ### Up to 3,388 pages have no trend_pct
    These are likely new pages with zero prior-period baseline, so this data can't tell us anything about trend/refresh-worthiness for brand new content.

2. ### Extreme outliers exist in this dataset
    The data has real, structural skew, so any score is sensitive to we clip/cap, and a few pages will always sit at the edges of what's 'normal'.

3. ### No casual info
     Correlations and freshness are'nt proof that refreshing causes improvement

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.